# Agentic RAG

# NEXT LEVEL

It's time to take our Agentic RAG further, using a Tool to supplement our MCP servers.

I'm working on this Expert in questions about my courses - but you should apply this to your domain!

I've put a bunch of questions in `knowledge/faq.jsonl`

Please see the README for setup instructions.

In [1]:
from agents import Agent, Runner, SQLiteSession, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
import json
from IPython.display import display, Markdown
from pathlib import Path
from qdrant_client import QdrantClient
import gradio as gr
load_dotenv(override=True)

True

In [3]:
model = "gpt-5.4-nano"

## The MCP Parameters

In [4]:
knowledge_dir = Path.cwd() / "knowledge"
knowledge_dir.mkdir(exist_ok=True)
vectordb_path = knowledge_dir / "vectordb"
faq_path = knowledge_dir / "faq.jsonl"

vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

In [ ]:
client = QdrantClient(path=str(vectordb_path))
collection_name = "knowledge"

info = client.get_collection(collection_name)
print(f"Memories in '{collection_name}': {info.points_count}\n")

points, _ = client.scroll(collection_name=collection_name, limit=200, with_payload=True, with_vectors=False)
for i, p in enumerate(points, 1):
    doc = (p.payload or {}).get("document", "")
    preview = doc.replace("\n", " ")[:160]
    print(f"{i:>3}. {preview}{'...' if len(doc) > 160 else ''}")

client.close()

In [ ]:
with faq_path.open() as f:
    faqs = [json.loads(line) for line in f]

In [ ]:
instructions = """
# Role

You are an expert about Ed Donner and his online courses. You are answering questions about him and his courses to visitors on his website.
Use your memories and tools to help find background information to answer the question. As needed, use multiple tools at the same time to gather all relevant context.
If you don't know the answer, say so.

# Memory

Always use your qdrant-find memory tool to help find relevant information. You can make multiple queries. Make all tool calls in parallel.

# FAQ

Your faq tool contains answers to all the common questions. Below is a list of the questions with their numbers.
If the user's question is related to one of these questions, then use your faq tool to retrieve a specific answer.
Respond with the answer in its original form in markdown, as written by Ed. If the answer include hyperlinks, then keep them in markdown format.

List of questions by number:
"""

for faq in faqs:
    instructions += f"\n{faq['faq']}. {faq['question']}"

In [ ]:
display(Markdown(instructions))

In [ ]:
faqs_lookup = {faq["faq"]: faq for faq in faqs}

def find_faq(question_number: int):
    faq = faqs_lookup.get(question_number)
    if faq:
        return f"### Question {faq['faq']} is:\n{faq['question']}\n### Ed's answer:\n{faq['answer']}"
    else:
        return "That question number was not found in the FAQ."   

@function_tool
def faq_tool(question_number:int ) -> str:
    """Use this tool to retrieve the answer to a frequently asked question by its number."""
    return find_faq(question_number)  

In [ ]:
faq_tool.description

In [ ]:
EXAMPLES = ["Which course covers RAG?", "Which course covers MCP?", "What job can I get after your courses?"]

In [ ]:
convo = SQLiteSession("test_conversation")

## New and improved Agentic RAG!

In [ ]:
def has_instant_answer(message: str):
    stripped = message.strip().lower()
    return stripped.startswith("q") and stripped[1:].isdigit() and len(stripped) <= 3

def get_instant_answer(message: str):
    question_number = int(message.strip()[1:])
    return find_faq(question_number)

In [ ]:
get_instant_answer("Q2")

In [ ]:
def text_from_event(event):
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        return event.data.delta
    elif event.type == "run_item_stream_event":
        if event.name == "tool_called":
            tool_name = getattr(event.item, "tool_name", None)
            return f'<small class="tool-status">Calling {tool_name}...</small>'
        elif event.name == "tool_output":
            return '<small class="tool-status">Tool returned. Thinking...</small>'

In [ ]:
async def run(message):
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
        cumulative = ""
        agent = Agent(name="Expert", model=model, instructions=instructions, tools=[faq_tool], mcp_servers=[vectorstore_mcp])
        response = Runner.run_streamed(agent, message, session=convo)
        async for event in response.stream_events():
            if (text := text_from_event(event)):
                cumulative += text
                yield cumulative

### Minor change

I made one small addition since the video - can you see the line I added, and do you see why I needed to do that?

In [ ]:
async def chat(message, history):
    if has_instant_answer(message):
        answer = get_instant_answer(message)
        await convo.add_items([{"role": "user", "content": message}, {"role": "assistant", "content": answer}])
        yield answer
    else:
        async for result in run(message):
            yield result

In [ ]:
from styles import CSS, JS
gr.ChatInterface(chat, examples=EXAMPLES, chatbot=gr.Chatbot(show_label=False, height=800)).launch(css=CSS, js=JS, theme=gr.themes.Base(), inbrowser=True)